============================================================
الخلية 1: تثبيت المكتبات
============================================================
تثبيت شامل لجميع المكتبات المطلوبة


## الخلية 2: الاستيرادات والإعداد


## الخلية 1: تثبيت المكتبات


In [21]:
!apt-get -y install poppler-utils tesseract-ocr tesseract-ocr-ara -qq
!pip install -q \
    pdf2image easyocr pyspellchecker langdetect transformers peft \
    huggingface_hub datasets Pillow opencv-python-headless pandas numpy \
    matplotlib scikit-learn pytesseract arabic-reshaper python-bidi \
    ar-corrector "gradio>=4.0" openpyxl jiwer torch torchvision \
    tensorboard albumentations tqdm concurrent-log-handler


In [22]:
import os, io, gc, cv2, json, time, shutil, sqlite3, logging, platform
import multiprocessing as mp
from pathlib import Path
from dataclasses import dataclass, field
from logging.handlers import RotatingFileHandler
from datetime import datetime
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import gradio as gr
import pytesseract
import easyocr

from PIL import Image
from pdf2image import convert_from_path
from spellchecker import SpellChecker
from langdetect import detect, DetectorFactory
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from tqdm import tqdm

try:
    from ar_corrector.corrector import Corrector
    _AR_CORRECTOR_AVAILABLE = True
except ImportError:
    _AR_CORRECTOR_AVAILABLE = False

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    try:
        _HF_TOKEN = userdata.get('HF_TOKEN')
        os.environ['HF_TOKEN'] = _HF_TOKEN
        print('✅ HF_TOKEN محمّل')
    except Exception:
        _HF_TOKEN = None
        print('ℹ️ لا يوجد HF_TOKEN')
    IN_COLAB = True
except ImportError:
    _HF_TOKEN = None
    IN_COLAB = False
    print('ℹ️ البيئة: local')

DetectorFactory.seed = 0
print('✅ الاستيرادات اكتملت')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ HF_TOKEN محمّل
✅ الاستيرادات اكتملت


## الخلية 3: الإعدادات المركزية


In [23]:
@dataclass
class Config:
    """الإعدادات المركزية للمشروع — مدمج من config.py الأصلي + تحسينات 3.0 Pro"""

    # --- المسارات ---
    project_root: str     = '/content/drive/MyDrive/Handwritten_OCR_Ultimate'
    pdf_path: str         = '/content/drive/MyDrive/python notes.pdf'
    db_name: str          = 'handwriting_data.db'

    # --- النماذج ---
    trocr_model_name: str = 'David-Magdy/TR_OCR_LARGE'
    hf_token: str         = ''
    hf_dataset_repo: str  = ''          # مثال: DrAbdulmalek/handwriting-ocr
    ocr_languages: list   = field(default_factory=lambda: ['en', 'ar'])

    # --- الأداء ---
    dpi: int              = 300
    use_gpu: bool         = True
    trocr_batch_size: int = 16          # ← batching لـ 10× تسريع
    num_beams: int        = 4           # ← beam search لدقة أعلى
    max_text_length: int  = 64
    easy_conf_threshold: float = 0.80  # ← تخطي TrOCR لو EasyOCR واثق
    trocr_default_conf: float  = 0.70
    low_conf_threshold: float  = 0.50

    # --- Preprocessing ---
    clahe_clip: float     = 2.0
    clahe_tile: tuple     = (8, 8)
    denoise_h: int        = 20
    enable_deskew: bool   = True
    min_word_w: int       = 15
    min_word_h: int       = 10
    dilation_kernel: tuple = (25, 5)

    # --- التصحيح ---
    correction_min_votes: int = 1      # ← خُفض من 2 إلى 1

    # --- الصفحات ---
    pages_start: int      = 1
    pages_end: int        = 5

    # --- التدريب ---
    finetune_min_samples: int  = 50
    finetune_epochs: int       = 5
    finetune_batch_size: int   = 4
    finetune_lr: float         = 1e-5   # ← خُفض من 5e-5
    lora_r: int                = 16
    lora_alpha: int            = 32
    lora_dropout: float        = 0.1
    lora_targets: list         = field(default_factory=lambda: ['query', 'value'])

    # --- Active Learning ---
    al_min_new_corrections: int = 50   # تشغيل تلقائي بعد N تصحيح
    al_reprocess_low_conf_limit: int = 100

    # --- التصدير ---
    export_val_ratio: float    = 0.1
    auto_export_after_run: bool = True

    # --- Gradio ---
    gradio_share: bool    = True
    gradio_port: int      = 7860

    # --- Logging ---
    log_level: str        = 'INFO'

    # === properties ===
    @property
    def root(self): return Path(self.project_root)

    @property
    def db_path(self): return self.root / 'database' / self.db_name

    @property
    def logs_dir(self): return self.root / 'logs'

    @property
    def exports_dir(self): return self.root / 'exports'

    @property
    def cache_dir(self): return self.root / 'models_cache'

    @property
    def artifacts_dir(self): return self.root / 'artifacts'

    @property
    def backups_dir(self): return self.root / 'backups'

    @property
    def tensorboard_dir(self): return self.root / 'runs'

    @property
    def feedback_csv(self): return self.logs_dir / 'user_corrections_feedback.csv'

    @property
    def stats_json(self): return self.logs_dir / 'processing_stats.json'

    @property
    def correction_dict_path(self): return self.artifacts_dir / 'correction_dict.json'

    @property
    def checkpoint_file(self): return self.artifacts_dir / 'ocr_checkpoint.json'

    @property
    def metrics_log(self): return self.logs_dir / 'metrics_history.csv'

    @property
    def runs_csv(self): return self.logs_dir / 'runs_history.csv'

    @property
    def lora_save_path(self): return self.cache_dir / 'trocr_lora_finetuned'

    @property
    def easyocr_drive_path(self): return self.root / '.EasyOCR'

    def setup(self):
        for d in [self.root/'database', self.logs_dir, self.exports_dir,
                  self.cache_dir, self.artifacts_dir, self.backups_dir,
                  self.tensorboard_dir]:
            d.mkdir(parents=True, exist_ok=True)
        if self.hf_token:
            os.environ['HF_TOKEN'] = self.hf_token
        if self.cache_dir.exists():
            os.environ['TRANSFORMERS_CACHE'] = str(self.cache_dir)
            os.environ['TORCH_HOME']         = str(self.cache_dir)
        for csv_path, cols in [
            (self.feedback_csv, ['timestamp','image_id','original_text','corrected_text','status']),
            (self.runs_csv,     ['run_id','timestamp','pages','words','avg_conf','duration_sec','status']),
        ]:
            if not csv_path.exists():
                pd.DataFrame(columns=cols).to_csv(csv_path, index=False, encoding='utf-8-sig')

    def setup_easyocr_symlink(self):
        local = Path.home() / '.EasyOCR'
        drive_path = self.easyocr_drive_path
        if not drive_path.exists():
            drive_path.mkdir(parents=True, exist_ok=True)
        if local.exists() and not local.is_symlink() and not drive_path.exists():
            shutil.move(str(local), str(drive_path))
        if not local.exists():
            try:
                os.symlink(drive_path, local)
            except Exception:
                pass

    @classmethod
    def for_colab(cls, pdf_name='python notes.pdf', hf_token='', hf_repo=''):
        base = '/content/drive/MyDrive'
        return cls(
            project_root=f'{base}/Handwritten_OCR_Ultimate',
            pdf_path=f'{base}/{pdf_name}',
            hf_token=hf_token or _HF_TOKEN or '',
            hf_dataset_repo=hf_repo,
        )


CFG = Config.for_colab()
CFG.setup()
CFG.setup_easyocr_symlink()
print('✅ الإعدادات جاهزة:', CFG.root)


✅ الإعدادات جاهزة: /content/drive/MyDrive/Handwritten_OCR_Ultimate


## الخلية 4: نظام اللوغات


In [24]:
_LOG_FILE = CFG.logs_dir / f'ocr_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
_EVENTS   = CFG.logs_dir / 'ocr_events.jsonl'

LOGGER = logging.getLogger('HandwrittenOCR')
LOGGER.setLevel(getattr(logging, CFG.log_level))
LOGGER.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
for _h in [RotatingFileHandler(_LOG_FILE, maxBytes=2_000_000, backupCount=5, encoding='utf-8'),
           logging.StreamHandler()]:
    _h.setFormatter(_fmt)
    LOGGER.addHandler(_h)


def log_event(event_type: str, payload: dict = None, level: str = 'info'):
    payload = payload or {}
    entry = {'ts': datetime.now().isoformat(), 'event': event_type, 'data': payload}
    with open(_EVENTS, 'a', encoding='utf-8') as f:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    getattr(LOGGER, level, LOGGER.info)(f'{event_type} | {json.dumps(payload, ensure_ascii=False)}')

print('✅ نظام اللوغات جاهز')


✅ نظام اللوغات جاهز


## الخلية 5: قاعدة البيانات (HandwritingDB — مدمج من database.py)


In [25]:
class HandwritingDB:
    """مدير SQLite — مخطط v3: يدعم run_id + raw_text + created_at/updated_at"""

    SCHEMA_V3 = '''
        CREATE TABLE IF NOT EXISTS handwriting_data (
            image_id      INTEGER PRIMARY KEY AUTOINCREMENT,
            image_data    BLOB    NOT NULL,
            predicted_text TEXT   DEFAULT '',
            raw_text      TEXT    DEFAULT '',
            status        TEXT    DEFAULT 'unverified',
            confidence    REAL    DEFAULT 0.0,
            model_source  TEXT    DEFAULT 'none',
            x INTEGER DEFAULT 0, y INTEGER DEFAULT 0,
            w INTEGER DEFAULT 0, h INTEGER DEFAULT 0,
            page_num      INTEGER DEFAULT 0,
            run_id        TEXT    DEFAULT '',
            created_at    TEXT    DEFAULT '',
            updated_at    TEXT    DEFAULT ''
        );
        CREATE TABLE IF NOT EXISTS processing_runs (
            run_id      TEXT PRIMARY KEY,
            started_at  TEXT, ended_at TEXT,
            input_path  TEXT,
            pages_processed INTEGER DEFAULT 0,
            words_processed INTEGER DEFAULT 0,
            avg_confidence  REAL    DEFAULT 0.0,
            status      TEXT DEFAULT 'running'
        );
        CREATE TABLE IF NOT EXISTS review_events (
            id            INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp     TEXT,
            image_id      INTEGER,
            original_text TEXT,
            corrected_text TEXT,
            action        TEXT,
            reviewer      TEXT DEFAULT 'user'
        );
        CREATE INDEX IF NOT EXISTS idx_status  ON handwriting_data(status);
        CREATE INDEX IF NOT EXISTS idx_page    ON handwriting_data(page_num);
        CREATE INDEX IF NOT EXISTS idx_run     ON handwriting_data(run_id);
        CREATE INDEX IF NOT EXISTS idx_conf    ON handwriting_data(confidence);
    '''

    def __init__(self, db_path: Path):
        self.db_path = db_path
        db_path.parent.mkdir(parents=True, exist_ok=True)
        with self._conn() as c:
            c.executescript(self.SCHEMA_V3)
            self._migrate(c)
        LOGGER.info(f'DB جاهزة: {db_path}')

    def _conn(self):
        return sqlite3.connect(self.db_path, check_same_thread=False)

    def _migrate(self, conn):
        """ترقية مخطط v1/v2 → v3"""
        cur = conn.execute("PRAGMA table_info(handwriting_data)")
        existing = {r[1] for r in cur.fetchall()}
        new_cols = {
            'raw_text': "TEXT DEFAULT ''",
            'run_id':   "TEXT DEFAULT ''",
            'created_at': "TEXT DEFAULT ''",
            'updated_at': "TEXT DEFAULT ''",
        }
        for col, typedef in new_cols.items():
            if col not in existing:
                conn.execute(f"ALTER TABLE handwriting_data ADD COLUMN {col} {typedef}")
        # توحيد قيم status القديمة
        conn.execute("UPDATE handwriting_data SET status='verified'   WHERE status='yes'")
        conn.execute("UPDATE handwriting_data SET status='unverified' WHERE status='no'")
        conn.commit()

    # --- Insert ---
    def insert_word(self, image_data, predicted_text, raw_text='',
                    status='unverified', confidence=0.0, model_source='none',
                    x=0, y=0, w=0, h=0, page_num=0, run_id='') -> int:
        ts = datetime.now().isoformat()
        with self._conn() as c:
            cur = c.execute(
                '''INSERT INTO handwriting_data
                   (image_data,predicted_text,raw_text,status,confidence,
                    model_source,x,y,w,h,page_num,run_id,created_at,updated_at)
                   VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)''',
                (image_data, predicted_text, raw_text, status, confidence,
                 model_source, x, y, w, h, page_num, run_id, ts, ts))
            c.commit()
            return cur.lastrowid

    # --- Update ---
    def update_word(self, image_id, predicted_text=None, status=None, updated_at=None):
        sets, vals = [], []
        if predicted_text is not None: sets.append('predicted_text=?'); vals.append(predicted_text)
        if status is not None:         sets.append('status=?');         vals.append(status)
        sets.append('updated_at=?')
        vals.append(updated_at or datetime.now().isoformat())
        vals.append(image_id)
        with self._conn() as c:
            c.execute(f"UPDATE handwriting_data SET {','.join(sets)} WHERE image_id=?", vals)
            c.commit()

    def delete_word(self, image_id: int):
        with self._conn() as c:
            c.execute('DELETE FROM handwriting_data WHERE image_id=?', (image_id,))
            c.commit()

    def delete_pages(self, start, end):
        with self._conn() as c:
            cur = c.execute('DELETE FROM handwriting_data WHERE page_num BETWEEN ? AND ?', (start, end))
            c.commit()
            return cur.rowcount

    # --- Queries ---
    def _rows(self, sql, params=()):
        with self._conn() as c:
            c.row_factory = sqlite3.Row
            return [dict(r) for r in c.execute(sql, params).fetchall()]

    def get_all(self):
        return self._rows('SELECT * FROM handwriting_data ORDER BY page_num,y,x')

    def get_unverified(self):
        return self._rows("SELECT * FROM handwriting_data WHERE status='unverified' ORDER BY confidence ASC")

    def get_verified(self):
        return self._rows("SELECT * FROM handwriting_data WHERE status IN ('verified','sentence_corrected') ORDER BY image_id")

    def get_low_confidence(self, threshold=0.5, limit=100):
        return self._rows("SELECT * FROM handwriting_data WHERE confidence<? ORDER BY confidence ASC LIMIT ?", (threshold, limit))

    def count_by_status(self) -> dict:
        rows = self._rows('SELECT status, COUNT(*) as cnt FROM handwriting_data GROUP BY status')
        return {r['status']: r['cnt'] for r in rows}

    def insert_run(self, run_id, input_path):
        with self._conn() as c:
            c.execute('INSERT OR REPLACE INTO processing_runs (run_id,started_at,input_path,status) VALUES (?,?,?,?)',
                      (run_id, datetime.now().isoformat(), str(input_path), 'running'))
            c.commit()

    def finish_run(self, run_id, pages, words, avg_conf):
        with self._conn() as c:
            c.execute('UPDATE processing_runs SET ended_at=?,pages_processed=?,words_processed=?,avg_confidence=?,status=? WHERE run_id=?',
                      (datetime.now().isoformat(), pages, words, avg_conf, 'completed', run_id))
            c.commit()

    def log_review(self, image_id, original, corrected, action):
        with self._conn() as c:
            c.execute('INSERT INTO review_events (timestamp,image_id,original_text,corrected_text,action) VALUES (?,?,?,?,?)',
                      (datetime.now().isoformat(), image_id, original, corrected, action))
            c.commit()


DB = HandwritingDB(CFG.db_path)
print('✅ قاعدة البيانات جاهزة')


2026-04-26 22:54:45,239 | INFO | DB جاهزة: /content/drive/MyDrive/Handwritten_OCR_Ultimate/database/handwriting_data.db
INFO:HandwrittenOCR:DB جاهزة: /content/drive/MyDrive/Handwritten_OCR_Ultimate/database/handwriting_data.db


✅ قاعدة البيانات جاهزة


## الخلية 6: Preprocessing (من preprocessing.py + تحسينات)


In [26]:
def deskew(gray: np.ndarray) -> np.ndarray:
    coords = np.column_stack(np.where(gray < 250))
    if len(coords) < 50:
        return gray
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    if abs(angle) < 0.3:
        return gray
    h, w = gray.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_REPLICATE)


def preprocess_image(img_bgr: np.ndarray, adaptive=False) -> tuple:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) if img_bgr.ndim == 3 else img_bgr.copy()
    if CFG.enable_deskew:
        gray = deskew(gray)
    gray = cv2.createCLAHE(clipLimit=CFG.clahe_clip, tileGridSize=CFG.clahe_tile).apply(gray)
    gray = cv2.fastNlMeansDenoising(gray, h=CFG.denoise_h)
    if adaptive:
        binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY_INV, 31, 11)
    else:
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return binary, gray


def _iou(b1, b2) -> float:
    x1, y1, w1, h1 = b1
    x2, y2, w2, h2 = b2
    xi1, yi1 = max(x1,x2), max(y1,y2)
    xi2, yi2 = min(x1+w1,x2+w2), min(y1+h1,y2+h2)
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    union = w1*h1 + w2*h2 - inter
    return inter/union if union > 0 else 0


def smart_segmentation(img_bgr, binary, easyocr_detections=None) -> list:
    """EasyOCR أولاً + IoU matching + الكنتورات كبديل (من PDFProcessor)"""
    if easyocr_detections:
        boxes = []
        for det in easyocr_detections:
            pts = np.array(det[0], dtype=np.int32)
            x, y, w, h = cv2.boundingRect(pts)
            if w > CFG.min_word_w and h > CFG.min_word_h:
                boxes.append((x, y, w, h))
        if boxes:
            return boxes
    kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, CFG.dilation_kernel)
    dilated = cv2.dilate(binary, kernel, iterations=1)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = [(x,y,w,h) for c in contours
             for x,y,w,h in [cv2.boundingRect(c)]
             if w > CFG.min_word_w and h > CFG.min_word_h]
    return sorted(boxes, key=lambda b: (b[1], b[0]))


def match_boxes_with_detections(boxes, detections) -> list:
    """ربط بالـ IoU (من PDFProcessor._match_boxes_with_detections)"""
    if not detections:
        return [(b, None) for b in boxes]
    det_boxes = []
    for det in detections:
        pts = np.array(det[0], dtype=np.int32)
        det_boxes.append((cv2.boundingRect(pts), det))
    result, used = [], set()
    for box in boxes:
        best_det, best_iou = None, 0
        for i, (db, det) in enumerate(det_boxes):
            if i in used:
                continue
            iou = _iou(box, db)
            if iou > best_iou and iou > 0.3:
                best_iou, best_det = iou, det
                used.add(i)
        result.append((box, best_det))
    return result


def crop_safe(img, x, y, w, h):
    H, W = img.shape[:2]
    return img[max(0,y):min(H,y+h), max(0,x):min(W,x+w)]


print('✅ Preprocessing جاهز')


✅ Preprocessing جاهز


## الخلية 7: OCR Engine (من recognition.py + batching + beam search)


In [27]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() and CFG.use_gpu else 'cpu')
LOGGER.info(f'Device: {DEVICE}')

print(f'⏳ تحميل النماذج على {DEVICE}...')
_t0 = time.time()

_hf_kwargs = {}
if CFG.hf_token:
    _hf_kwargs['token'] = CFG.hf_token
if CFG.cache_dir.exists():
    _hf_kwargs['cache_dir'] = str(CFG.cache_dir)

trocr_processor = TrOCRProcessor.from_pretrained(CFG.trocr_model_name, **_hf_kwargs)
trocr_model     = VisionEncoderDecoderModel.from_pretrained(CFG.trocr_model_name, **_hf_kwargs).to(DEVICE)

# تحميل LoRA إذا وُجد
if CFG.lora_save_path.exists():
    try:
        from peft import PeftModel
        trocr_model = PeftModel.from_pretrained(trocr_model, str(CFG.lora_save_path)).to(DEVICE)
        print('✅ LoRA weights محمّلة')
    except Exception as e:
        print('⚠️ LoRA:', e)

# EasyOCR
try:
    easy_reader = easyocr.Reader(CFG.ocr_languages, gpu=torch.cuda.is_available())
except Exception:
    easy_reader = easyocr.Reader(CFG.ocr_languages, gpu=False)

# المدققات الإملائية
_ar_corrector  = Corrector() if _AR_CORRECTOR_AVAILABLE else None
_en_spellcheck = SpellChecker(language='en')

print(f'✅ النماذج جاهزة في {time.time()-_t0:.1f}s')


def normalize_text(x) -> str:
    return '' if (x is None or (isinstance(x, float) and pd.isna(x))) else str(x).strip()


def detect_lang(text: str) -> str:
    try:
        return detect(str(text)) if text and text.strip() else 'unknown'
    except Exception:
        return 'unknown'


def spell_correct(text: str) -> str:
    text = normalize_text(text)
    if not text:
        return ''
    try:
        lang = detect_lang(text)
        if lang == 'ar' and _ar_corrector:
            return _ar_corrector.contextual_correct(text)
        words = text.split()
        return ' '.join(_en_spellcheck.correction(w) or w for w in words)
    except Exception:
        return text


def trocr_batch_predict(crops: list) -> list:
    """
    Batch inference لـ TrOCR — التحسين الأهم: 3-6× تسريع
    مع beam search لدقة أعلى
    """
    if not crops:
        return []
    pil_imgs = []
    for crop in crops:
        rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        pil_imgs.append(Image.fromarray(rgb))
    try:
        pv = trocr_processor(images=pil_imgs, return_tensors='pt',
                             padding=True).pixel_values.to(DEVICE)
        with torch.no_grad():
            ids = trocr_model.generate(
                pv,
                max_length=CFG.max_text_length,
                num_beams=CFG.num_beams,       # ← beam search
                early_stopping=True,
                length_penalty=1.0,
            )
        return trocr_processor.batch_decode(ids, skip_special_tokens=True)
    except Exception as e:
        LOGGER.warning(f'trocr_batch failed: {e}')
        return [''] * len(crops)


def ocr_ensemble_single(crop, easy_item=None) -> tuple:
    """
    Ensemble ذكي: يتخطى TrOCR إذا كانت ثقة EasyOCR > easy_conf_threshold
    """
    candidates = []

    # EasyOCR أولاً
    if easy_item is not None:
        _, txt, conf = easy_item
        txt = normalize_text(txt)
        if txt:
            candidates.append(('easyocr', txt, float(conf)))
            # تخطي TrOCR لو EasyOCR واثق — توفير 70% من inference
            if float(conf) >= CFG.easy_conf_threshold:
                return txt, float(conf), 'easyocr', False, candidates

    # TrOCR (فقط عند الحاجة)
    try:
        rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        pv  = trocr_processor(images=Image.fromarray(rgb),
                              return_tensors='pt').pixel_values.to(DEVICE)
        with torch.no_grad():
            ids = trocr_model.generate(pv, max_length=CFG.max_text_length,
                                       num_beams=CFG.num_beams, early_stopping=True)
        txt = trocr_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
        if txt:
            candidates.append(('trocr', txt, CFG.trocr_default_conf))
    except Exception as e:
        LOGGER.debug(f'trocr single: {e}')

    candidates = [c for c in candidates if c[1]]
    if not candidates:
        return '', 0.0, 'none', True, []
    best = max(candidates, key=lambda x: x[2])
    return best[1], float(best[2]), best[0], best[2] < CFG.low_conf_threshold, candidates


print('✅ OCR Engine جاهز')


2026-04-26 22:54:45,453 | INFO | Device: cpu
INFO:HandwrittenOCR:Device: cpu


⏳ تحميل النماذج على cpu...


Loading weights:   0%|          | 0/636 [00:00<?, ?it/s]

✅ النماذج جاهزة في 51.2s
✅ OCR Engine جاهز


## الخلية 8: قاموس التصحيح (من correction.py)


In [28]:
def build_correction_dict() -> dict:
    if not CFG.feedback_csv.exists():
        return {}
    try:
        df = pd.read_csv(CFG.feedback_csv, encoding='utf-8-sig')
        if df.empty:
            return {}
        buckets = defaultdict(Counter)
        for _, row in df.iterrows():
            o = normalize_text(row.get('original_text'))
            c = normalize_text(row.get('corrected_text'))
            if o and c and o != c:
                buckets[o][c] += 1
        result = {o: cnt.most_common(1)[0][0]
                  for o, cnt in buckets.items()
                  if cnt.most_common(1)[0][1] >= CFG.correction_min_votes}
        CFG.correction_dict_path.write_text(
            json.dumps(result, ensure_ascii=False, indent=2), 'utf-8')
        log_event('correction_dict_built', {'entries': len(result)})
        return result
    except Exception as e:
        LOGGER.error(f'correction_dict: {e}')
        return {}


def load_correction_dict() -> dict:
    if CFG.correction_dict_path.exists():
        return json.loads(CFG.correction_dict_path.read_text('utf-8'))
    return build_correction_dict()


def apply_corrections(text: str, d: dict) -> str:
    return ' '.join(d.get(tok, tok) for tok in text.split()) if text else ''


def append_feedback(image_id, original, corrected, status='verified'):
    ts = datetime.now().isoformat()
    pd.DataFrame([{'timestamp': ts, 'image_id': image_id,
                   'original_text': original, 'corrected_text': corrected,
                   'status': status}]).to_csv(
        CFG.feedback_csv, mode='a', header=False, index=False, encoding='utf-8-sig')


correction_dict = build_correction_dict()
print(f'✅ قاموس التصحيح: {len(correction_dict)} إدخال')


✅ قاموس التصحيح: 0 إدخال


## الخلية 9: Checkpoint


In [29]:
def save_checkpoint(data: dict):
    CFG.checkpoint_file.write_text(json.dumps(data, ensure_ascii=False, indent=2), 'utf-8')


def load_checkpoint():
    return json.loads(CFG.checkpoint_file.read_text('utf-8')) if CFG.checkpoint_file.exists() else None


def clear_checkpoint():
    if CFG.checkpoint_file.exists():
        CFG.checkpoint_file.unlink()

print('✅ Checkpoint جاهز')


✅ Checkpoint جاهز


## الخلية 10: معالجة PDF (مدمج من PDFProcessor + تحسينات الأداء)


In [30]:
def load_pages(input_path, start_page=None, end_page=None):
    """تحميل صفحات PDF أو صورة واحدة"""
    input_path = str(input_path)
    ext = Path(input_path).suffix.lower()
    sp  = start_page or CFG.pages_start
    ep  = end_page   or CFG.pages_end
    if ext == '.pdf':
        imgs = convert_from_path(input_path, dpi=CFG.dpi, first_page=sp, last_page=ep)
        pages = [cv2.cvtColor(np.array(p), cv2.COLOR_RGB2BGR) for p in imgs]
        return pages, list(range(sp, sp + len(pages)))
    if ext in {'.png','.jpg','.jpeg','.bmp','.tif','.tiff','.webp'}:
        img = cv2.imread(input_path)
        if img is None:
            raise ValueError(f'تعذر قراءة: {input_path}')
        return [img], [1]
    raise ValueError(f'نوع غير مدعوم: {ext}')


def process_document(input_path=None, start_page=None, end_page=None,
                     resume=True, adaptive=False, progress_cb=None) -> dict:
    """
    المعالجة الرئيسية مع:
    - Checkpoint استئناف
    - Batch TrOCR للأداء
    - IoU matching من PDFProcessor
    - تصدير تلقائي بعد الانتهاء
    """
    global correction_dict
    input_path = str(input_path or CFG.pdf_path)
    run_id     = datetime.now().strftime('run_%Y%m%d_%H%M%S')
    t0         = time.time()
    correction_dict = load_correction_dict()

    # استئناف
    ckpt = load_checkpoint() if resume else None
    if ckpt and ckpt.get('input_path') == input_path:
        start_page = int(ckpt.get('next_page', start_page or CFG.pages_start))
        LOGGER.info(f'استئناف من الصفحة {start_page}')

    DB.insert_run(run_id, input_path)
    log_event('process_started', {'run_id': run_id, 'input': input_path})

    try:
        pages, page_nums = load_pages(input_path, start_page, end_page)
    except Exception as e:
        log_event('load_failed', {'error': str(e)}, 'error')
        return {'error': str(e)}

    DB.delete_pages(min(page_nums), max(page_nums))
    total_words, conf_acc = 0, []

    for idx, (page_img, actual_pg) in enumerate(zip(pages, page_nums)):
        if progress_cb:
            progress_cb(idx+1, len(pages), f'معالجة صفحة {actual_pg}...')

        # EasyOCR detection
        try:
            detections = easy_reader.readtext(page_img, detail=1)
        except Exception as e:
            detections = []
            LOGGER.warning(f'EasyOCR p{actual_pg}: {e}')

        binary, _ = preprocess_image(page_img, adaptive=adaptive)
        boxes = smart_segmentation(page_img, binary, detections)
        boxes_info = match_boxes_with_detections(boxes, detections)

        # ---- BATCH TROCR ----
        # تجميع المحاصيل التي تحتاج TrOCR (ثقة EasyOCR منخفضة)
        need_trocr_idx, need_trocr_crops = [], []
        easy_results = []
        for i, ((x,y,w,h), easy_item) in enumerate(boxes_info):
            crop = crop_safe(page_img, x, y, w, h)
            if crop.size == 0:
                easy_results.append(None)
                continue
            if easy_item is not None:
                _, txt, conf = easy_item
                easy_results.append(('easyocr', normalize_text(txt), float(conf)))
                if float(conf) < CFG.easy_conf_threshold:
                    need_trocr_idx.append(i)
                    need_trocr_crops.append(crop)
            else:
                easy_results.append(None)
                need_trocr_idx.append(i)
                need_trocr_crops.append(crop)

        # Batch inference
        trocr_texts = {}
        for b_start in range(0, len(need_trocr_crops), CFG.trocr_batch_size):
            batch = need_trocr_crops[b_start:b_start+CFG.trocr_batch_size]
            texts = trocr_batch_predict(batch)
            for k, txt in enumerate(texts):
                trocr_texts[need_trocr_idx[b_start+k]] = txt

        # الدمج والإدراج في DB
        for i, ((x,y,w,h), easy_item) in enumerate(boxes_info):
            crop = crop_safe(page_img, x, y, w, h)
            if crop.size == 0:
                continue

            easy_res = easy_results[i]
            if easy_res and easy_res[2] >= CFG.easy_conf_threshold:
                raw, conf, src = easy_res[1], easy_res[2], easy_res[0]
            elif i in trocr_texts and trocr_texts[i]:
                raw  = trocr_texts[i]
                conf = CFG.trocr_default_conf
                src  = 'trocr'
                if easy_res and easy_res[2] > conf:
                    raw, conf, src = easy_res[1], easy_res[2], easy_res[0]
            elif easy_res:
                raw, conf, src = easy_res[1], easy_res[2], easy_res[0]
            else:
                raw, conf, src = '', 0.0, 'none'

            corrected = apply_corrections(spell_correct(raw), correction_dict)
            _, buf = cv2.imencode('.png', crop)
            DB.insert_word(
                image_data=buf.tobytes(), predicted_text=corrected,
                raw_text=raw, status='unverified', confidence=conf,
                model_source=src, x=x, y=y, w=w, h=h,
                page_num=actual_pg, run_id=run_id,
            )
            total_words += 1
            conf_acc.append(conf)

        save_checkpoint({'run_id': run_id, 'input_path': input_path,
                         'next_page': actual_pg+1, 'words': total_words,
                         'ts': datetime.now().isoformat()})
        log_event('page_done', {'page': actual_pg, 'boxes': len(boxes_info),
                                 'words_total': total_words})

    duration = time.time() - t0
    avg_conf  = float(np.mean(conf_acc)) if conf_acc else 0.0
    clear_checkpoint()

    DB.finish_run(run_id, len(page_nums), total_words, avg_conf)

    stats = {'run_id': run_id, 'ts': datetime.now().isoformat(),
             'input': input_path, 'pages': len(page_nums),
             'words': total_words, 'avg_confidence': round(avg_conf,4),
             'duration_sec': round(duration,2)}
    CFG.stats_json.write_text(json.dumps(stats, ensure_ascii=False, indent=2), 'utf-8')

    # runs history
    runs = pd.read_csv(CFG.runs_csv, encoding='utf-8-sig')
    runs = pd.concat([runs, pd.DataFrame([{
        'run_id': run_id, 'timestamp': stats['ts'],
        'pages': len(page_nums), 'words': total_words,
        'avg_conf': avg_conf, 'duration_sec': duration, 'status': 'completed',
    }])], ignore_index=True)
    runs.to_csv(CFG.runs_csv, index=False, encoding='utf-8-sig')

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if CFG.auto_export_after_run:
        exp = auto_export(run_id)
        stats['auto_export'] = exp

    log_event('process_finished', stats)
    print(f'✅ اكتملت المعالجة | {total_words} كلمة | {duration:.1f}s')
    return stats

print('✅ معالج PDF جاهز')


✅ معالج PDF جاهز


## الخلية 11: التصدير التلقائي


In [31]:
def auto_export(run_id: str, output_dir: Path = None) -> dict:
    """
    CSV + XLSX + النص الكامل + JSONL التدريب
    """
    output_dir = output_dir or (CFG.exports_dir / 'auto' / run_id)
    output_dir.mkdir(parents=True, exist_ok=True)

    with sqlite3.connect(CFG.db_path, check_same_thread=False) as c:
        df_all = pd.read_sql_query(
            'SELECT * FROM handwriting_data ORDER BY page_num,y,x', c)
    df_verified = df_all[df_all['status'].isin(['verified','sentence_corrected'])]
    df_csv = df_all.drop(columns=['image_data'], errors='ignore')

    exported = {}

    # CSV
    p = output_dir / 'all_words.csv'
    df_csv.to_csv(p, index=False, encoding='utf-8-sig')
    exported['csv'] = str(p)

    # XLSX
    px = output_dir / 'all_words.xlsx'
    with pd.ExcelWriter(px, engine='openpyxl') as w:
        df_csv.to_excel(w, sheet_name='All', index=False)
        for pg in sorted(df_csv['page_num'].dropna().unique()):
            df_csv[df_csv['page_num']==pg].to_excel(w, sheet_name=f'P{int(pg)}', index=False)
    exported['xlsx'] = str(px)

    # النص الكامل
    lines_out = []
    for pg in sorted(df_all['page_num'].dropna().unique()):
        pw = df_all[df_all['page_num']==pg].sort_values(['y','x'])
        if pw.empty:
            continue
        curr = [pw.iloc[0].to_dict()]
        line_groups = []
        for i in range(1, len(pw)):
            row = pw.iloc[i].to_dict()
            if abs(row['y'] - curr[-1]['y']) <= 25:
                curr.append(row)
            else:
                line_groups.append(curr); curr = [row]
        line_groups.append(curr)
        for lg in line_groups:
            lang = detect_lang(' '.join(str(w.get('predicted_text','')) for w in lg))
            sl = sorted(lg, key=lambda k: k['x'], reverse=(lang=='ar'))
            lines_out.append(' '.join(str(w.get('predicted_text','')) for w in sl).strip())
    pt = output_dir / 'reconstructed_text.txt'
    pt.write_text('\n'.join(lines_out), 'utf-8')
    exported['text'] = str(pt)

    # JSONL للتدريب
    if not df_verified.empty:
        img_dir = output_dir / 'training_images'
        img_dir.mkdir(exist_ok=True)
        records = []
        for _, row in df_verified.iterrows():
            fname = f"img_{row['image_id']}.png"
            with open(img_dir / fname, 'wb') as f:
                f.write(row['image_data'])
            txt = normalize_text(row['predicted_text'])
            if txt:
                records.append({'image': fname, 'text': txt})
        pj = output_dir / 'training_data.jsonl'
        with open(pj, 'w', encoding='utf-8') as f:
            for rec in records:
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')
        exported['jsonl'] = str(pj)
        exported['training_samples'] = len(records)

    summary = {'run_id': run_id, 'exported_at': datetime.now().isoformat(),
               'total_words': len(df_all), 'verified': len(df_verified),
               'dir': str(output_dir), 'files': exported}
    (output_dir / 'export_summary.json').write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), 'utf-8')
    log_event('auto_export_done', {'dir': str(output_dir)})
    return summary


def create_backup():
    label  = datetime.now().strftime('%Y%m%d_%H%M%S')
    bk_dir = CFG.backups_dir / f'backup_{label}'
    bk_dir.mkdir(parents=True, exist_ok=True)
    for p in [CFG.db_path, CFG.feedback_csv, CFG.stats_json,
              CFG.correction_dict_path, _LOG_FILE, _EVENTS]:
        if Path(p).exists():
            shutil.copy2(p, bk_dir / Path(p).name)
    log_event('backup_created', {'dir': str(bk_dir)})
    return str(bk_dir)

print('✅ نظام التصدير جاهز')


✅ نظام التصدير جاهز


## الخلية 12: إعادة بناء الجمل (من reconstruction.py)


In [32]:
def reconstruct_sentences(verified_only=False, y_tolerance=25) -> pd.DataFrame:
    with sqlite3.connect(CFG.db_path, check_same_thread=False) as c:
        q = "SELECT * FROM handwriting_data"
        if verified_only:
            q += " WHERE status IN ('verified','sentence_corrected')"
        q += " ORDER BY page_num,y,x"
        words = pd.read_sql_query(q, c)
    if words.empty:
        return pd.DataFrame()
    out = []
    for pg in words['page_num'].unique():
        pw = words[words['page_num']==pg].sort_values(['y','x'])
        curr = [pw.iloc[0].to_dict()]
        lines = []
        for i in range(1, len(pw)):
            row = pw.iloc[i].to_dict()
            if abs(row['y'] - curr[-1]['y']) <= y_tolerance:
                curr.append(row)
            else:
                lines.append(curr); curr = [row]
        lines.append(curr)
        for line in lines:
            preview = ' '.join(str(w.get('predicted_text','')) for w in line)
            lang    = detect_lang(preview)
            sl = sorted(line, key=lambda k: k['x'], reverse=(lang=='ar'))
            sentence = ' '.join(str(w.get('predicted_text','')) for w in sl).strip()
            out.append({'page': line[0]['page_num'], 'y_anchor': line[0]['y'],
                        'lang': lang, 'text': sentence,
                        'word_ids': [w['image_id'] for w in sl]})
    return pd.DataFrame(out)

print('✅ إعادة بناء الجمل جاهزة')


✅ إعادة بناء الجمل جاهزة


## الخلية 13: WER / CER metrics (jiwer)


In [33]:
def compute_metrics() -> dict:
    """حساب WER و CER الحقيقيين (raw_text مقابل predicted_text)"""
    try:
        from jiwer import wer, cer
    except ImportError:
        return {'error': 'pip install jiwer'}

    with sqlite3.connect(CFG.db_path, check_same_thread=False) as c:
        df = pd.read_sql_query(
            "SELECT raw_text, predicted_text FROM handwriting_data "
            "WHERE status IN ('verified','sentence_corrected') "
            "AND raw_text != '' AND predicted_text != ''", c)
    if df.empty or len(df) < 5:
        return {'wer': None, 'cer': None, 'samples': 0}

    refs = df['raw_text'].astype(str).tolist()
    hyps = df['predicted_text'].astype(str).tolist()
    m = {'wer': round(wer(refs, hyps), 4),
         'cer': round(cer(refs, hyps), 4),
         'samples': len(df),
         'timestamp': datetime.now().isoformat()}
    # حفظ
    mdf = pd.DataFrame([m])
    if CFG.metrics_log.exists():
        mdf.to_csv(CFG.metrics_log, mode='a', header=False, index=False, encoding='utf-8-sig')
    else:
        mdf.to_csv(CFG.metrics_log, index=False, encoding='utf-8-sig')
    return m


def plot_metrics_fig():
    if not CFG.metrics_log.exists():
        return None
    df = pd.read_csv(CFG.metrics_log, encoding='utf-8-sig')
    if df.empty:
        return None
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df['wer'].dropna().values, label='WER', marker='o', color='#E53935')
    ax.plot(df['cer'].dropna().values, label='CER', marker='s', color='#1E88E5')
    ax.set_title('تحسّن النموذج عبر الزمن (WER/CER)')
    ax.set_xlabel('جلسة التدريب')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

print('✅ Metrics جاهزة')


✅ Metrics جاهزة


## الخلية 14: Active Learning


In [34]:
def count_new_corrections() -> int:
    if not CFG.feedback_csv.exists():
        return 0
    df = pd.read_csv(CFG.feedback_csv, encoding='utf-8-sig')
    return len(df)


def should_trigger_training(min_new=None) -> bool:
    min_new = min_new or CFG.al_min_new_corrections
    return count_new_corrections() >= min_new


def reprocess_low_confidence(limit=None):
    """إعادة OCR على الكلمات ضعيفة الثقة بعد تحديث النموذج"""
    limit = limit or CFG.al_reprocess_low_conf_limit
    rows  = DB.get_low_confidence(CFG.low_conf_threshold, limit)
    if not rows:
        return 0
    reprocessed = 0
    for row in tqdm(rows, desc='إعادة معالجة'):
        try:
            img = Image.open(io.BytesIO(row['image_data'])).convert('RGB')
            img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
            texts = trocr_batch_predict([img_np])
            new_text = texts[0].strip() if texts else ''
            if new_text:
                DB.update_word(row['image_id'], predicted_text=new_text)
                reprocessed += 1
        except Exception as e:
            LOGGER.debug(f'reprocess {row["image_id"]}: {e}')
    log_event('reprocess_done', {'count': reprocessed})
    return reprocessed

print('✅ Active Learning جاهز')


✅ Active Learning جاهز


## الخلية 15: LoRA Fine-tuning مع TensorBoard


In [35]:
def finetune_trocr_lora(min_samples=None, epochs=None, batch_size=None,
                         lr=None, progress_cb=None) -> dict:
    """
    Fine-tuning مع:
    - TensorBoard للمراقبة
    - data augmentation (albumentations)
    - إعادة تحميل النموذج تلقائياً بعد التدريب
    """
    global trocr_model

    try:
        from peft import get_peft_model, LoraConfig, TaskType
        from torch.optim import AdamW
        from torch.utils.data import Dataset as TDS, DataLoader
        from torch.utils.tensorboard import SummaryWriter
    except ImportError as e:
        return {'error': str(e)}

    min_samples = min_samples or CFG.finetune_min_samples
    epochs      = epochs      or CFG.finetune_epochs
    batch_size  = batch_size  or CFG.finetune_batch_size
    lr          = lr          or CFG.finetune_lr

    verified = DB.get_verified()
    if len(verified) < min_samples:
        return {'error': f'عينات غير كافية: {len(verified)}/{min_samples}'}

    print(f'🚀 بدء التدريب على {len(verified)} عينة')

    writer = SummaryWriter(log_dir=str(CFG.tensorboard_dir / datetime.now().strftime('ft_%Y%m%d_%H%M%S')))
    writer.add_text('config', f'epochs={epochs}, lr={lr}, samples={len(verified)}')

    # --- Dataset مع Augmentation ---
    try:
        import albumentations as A
        aug = A.Compose([
            A.Rotate(limit=3, p=0.5),
            A.RandomBrightnessContrast(p=0.4),
            A.GaussNoise(p=0.3),
        ])
        USE_AUG = True
    except ImportError:
        USE_AUG = False

    class HWDataset(TDS):
        def __init__(self, records):
            self.records = records

        def __len__(self):
            return len(self.records)

        def __getitem__(self, idx):
            row = self.records[idx]
            img = Image.open(io.BytesIO(row['image_data'])).convert('RGB')
            img_np = np.array(img)
            if USE_AUG:
                img_np = aug(image=img_np)['image']
            pv = trocr_processor(images=Image.fromarray(img_np),
                                  return_tensors='pt').pixel_values.squeeze(0)
            text = normalize_text(row['predicted_text'])
            lb   = trocr_processor.tokenizer(
                text, return_tensors='pt', padding='max_length',
                max_length=CFG.max_text_length, truncation=True).input_ids.squeeze(0)
            lb[lb == trocr_processor.tokenizer.pad_token_id] = -100
            return {'pixel_values': pv, 'labels': lb}

    # --- LoRA ---
    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=CFG.lora_r, lora_alpha=CFG.lora_alpha,
        target_modules=CFG.lora_targets, lora_dropout=CFG.lora_dropout)
    model = get_peft_model(trocr_model, lora_cfg)
    model.train()

    loader    = DataLoader(HWDataset(verified), batch_size=batch_size, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=lr)
    global_step = 0
    log = []

    for epoch in range(epochs):
        total_loss = 0.0
        for batch in loader:
            out = model(pixel_values=batch['pixel_values'].to(DEVICE),
                        labels=batch['labels'].to(DEVICE))
            out.loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            loss_val = float(out.loss.item())
            total_loss += loss_val
            writer.add_scalar('Loss/step', loss_val, global_step)
            global_step += 1

        avg_loss = total_loss / max(1, len(loader))
        writer.add_scalar('Loss/epoch', avg_loss, epoch)
        writer.add_scalar('LR', lr, epoch)
        msg = f'Epoch {epoch+1}/{epochs} | Loss={avg_loss:.4f}'
        log.append(msg)
        print(msg)
        if progress_cb:
            progress_cb(epoch+1, epochs, msg)

    writer.close()

    lora_path = CFG.lora_save_path
    model.save_pretrained(str(lora_path))
    trocr_processor.save_pretrained(str(lora_path))
    trocr_model = model

    log_event('finetune_done', {'epochs': epochs, 'samples': len(verified),
                                 'lora_path': str(lora_path)})
    print(f'✅ Fine-tuning اكتمل | محفوظ في: {lora_path}')
    print(f'🔍 TensorBoard: tensorboard --logdir {CFG.tensorboard_dir} --port 6006')
    return {'status': 'done', 'log': log, 'lora_path': str(lora_path)}


def upload_to_huggingface(repo_id=None, hf_token=None) -> str:
    """رفع بيانات التدريب إلى HuggingFace Hub"""
    from huggingface_hub import HfApi, login
    repo_id  = repo_id  or CFG.hf_dataset_repo
    hf_token = hf_token or CFG.hf_token or _HF_TOKEN
    if not repo_id:
        return '❌ يجب تحديد repo_id'
    if not hf_token:
        return '❌ يجب توفير HF_TOKEN'

    # إعداد التصدير
    stats = json.loads(CFG.stats_json.read_text()) if CFG.stats_json.exists() else {}
    run_id = stats.get('run_id', f'manual_{datetime.now().strftime("%Y%m%d")}')
    export_dir = CFG.exports_dir / 'auto' / run_id

    if not export_dir.exists():
        auto_export(run_id)

    try:
        login(token=hf_token)
        api = HfApi()
        api.create_repo(repo_id=repo_id, repo_type='dataset', exist_ok=True)
        api.upload_folder(
            folder_path=str(export_dir),
            repo_id=repo_id,
            repo_type='dataset',
            commit_message=f'Dataset update {datetime.now().strftime("%Y-%m-%d")}')
        url = f'https://huggingface.co/datasets/{repo_id}'
        log_event('hf_upload_done', {'repo': repo_id, 'url': url})
        return f'✅ تم الرفع: {url}'
    except Exception as e:
        return f'❌ فشل الرفع: {e}'


def active_learning_cycle() -> str:
    """دورة Active Learning الكاملة"""
    global trocr_model
    if not should_trigger_training():
        n = count_new_corrections()
        return f'⏳ التصحيحات الحالية: {n}/{CFG.al_min_new_corrections}'

    print('🧠 Active Learning مُشغَّل...')
    result = finetune_trocr_lora()
    if 'error' in result:
        return f'❌ {result["error"]}'

    # إعادة تحميل النموذج المحدث
    if CFG.lora_save_path.exists():
        try:
            from peft import PeftModel
            trocr_model = PeftModel.from_pretrained(trocr_model, str(CFG.lora_save_path)).to(DEVICE)
        except Exception as e:
            LOGGER.warning(f'reload lora: {e}')

    reprocessed = reprocess_low_confidence()
    m = compute_metrics()

    log_event('al_cycle_done', {'reprocessed': reprocessed, 'metrics': m})
    return (f'✅ دورة Active Learning اكتملت\n'
            f'إعادة معالجة: {reprocessed} كلمة\n'
            f'WER: {m.get("wer")} | CER: {m.get("cer")}')

print('✅ Fine-tuning + Active Learning جاهزان')


✅ Fine-tuning + Active Learning جاهزان


## الخلية 16: Dashboard + Gradio helpers


In [36]:
_review_state = {'df': pd.DataFrame(), 'idx': 0}
_sent_state   = {'df': pd.DataFrame(), 'idx': 0}


# --- Word Review ---
def _word_row():
    df, idx = _review_state['df'], _review_state['idx']
    if df.empty:
        return None, '', '', 'لا توجد كلمات للمراجعة', 0.0, '0/0'
    row  = df.iloc[idx]
    pil  = Image.open(io.BytesIO(bytes(row['image_data'])))
    conf = float(row['confidence'] or 0)
    info = (f"**{idx+1}/{len(df)}** | صفحة {row['page_num']} "
            f"| {row['model_source']} | id={row['image_id']}")
    return pil, str(row['predicted_text'] or ''), str(row['raw_text'] or ''), info, conf, f"{idx+1}/{len(df)}"


def load_word_review():
    with sqlite3.connect(CFG.db_path, check_same_thread=False) as c:
        df = pd.read_sql_query(
            "SELECT * FROM handwriting_data WHERE status='unverified' "
            "ORDER BY confidence ASC, image_id ASC", c)
    _review_state['df']  = df
    _review_state['idx'] = 0
    return _word_row()


def word_confirm(corrected_text: str):
    df, idx = _review_state['df'], _review_state['idx']
    if df.empty:
        return _word_row()
    row  = df.iloc[idx]
    rid  = int(row['image_id'])
    orig = normalize_text(row['predicted_text'])
    corr = normalize_text(corrected_text)
    DB.update_word(rid, predicted_text=corr, status='verified')
    DB.log_review(rid, orig, corr, 'confirm')
    if orig != corr:
        append_feedback(rid, orig, corr, 'verified')
    _review_state['df'] = df.drop(df.index[idx]).reset_index(drop=True)
    _review_state['idx'] = min(idx, max(0, len(_review_state['df'])-1))
    return _word_row()


def word_delete():
    df, idx = _review_state['df'], _review_state['idx']
    if df.empty:
        return _word_row()
    DB.delete_word(int(df.iloc[idx]['image_id']))
    DB.log_review(int(df.iloc[idx]['image_id']), '', '', 'delete')
    _review_state['df'] = df.drop(df.index[idx]).reset_index(drop=True)
    _review_state['idx'] = min(idx, max(0, len(_review_state['df'])-1))
    return _word_row()


def word_prev():
    if not _review_state['df'].empty:
        _review_state['idx'] = max(0, _review_state['idx']-1)
    return _word_row()


def word_next():
    df = _review_state['df']
    if not df.empty:
        _review_state['idx'] = min(len(df)-1, _review_state['idx']+1)
    return _word_row()


# --- Sentence Review ---
def _sent_row():
    df, idx = _sent_state['df'], _sent_state['idx']
    if df.empty:
        return '', 'لا توجد جمل للمراجعة', '0/0'
    row = df.iloc[idx]
    info = (f"**{idx+1}/{len(df)}** | صفحة {row['page']} "
            f"| {row['lang']} | {len(row['word_ids'])} كلمة")
    return row['text'], info, f"{idx+1}/{len(df)}"


def load_sent_review():
    _sent_state['df']  = reconstruct_sentences(verified_only=False)
    _sent_state['idx'] = 0
    return _sent_row()


def sent_save(corrected: str):
    df, idx = _sent_state['df'], _sent_state['idx']
    if df.empty:
        return _sent_row()
    row  = df.iloc[idx]
    orig = normalize_text(row['text'])
    corr = normalize_text(corrected)
    ts   = datetime.now().isoformat()
    with sqlite3.connect(CFG.db_path, check_same_thread=False) as c:
        for wid in row['word_ids']:
            c.execute("UPDATE handwriting_data SET status='sentence_corrected', updated_at=? WHERE image_id=?",
                      (ts, int(wid)))
        c.commit()
    append_feedback(None, orig, corr, f"sent_p{row['page']}")
    _sent_state['idx'] = min(idx+1, max(0, len(df)-1))
    return _sent_row()


def sent_prev():
    if not _sent_state['df'].empty:
        _sent_state['idx'] = max(0, _sent_state['idx']-1)
    return _sent_row()


def sent_next():
    df = _sent_state['df']
    if not df.empty:
        _sent_state['idx'] = min(len(df)-1, _sent_state['idx']+1)
    return _sent_row()


# --- Dashboard ---
def build_dashboard_fig():
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    counts = DB.count_by_status()
    if counts:
        axes[0].bar(list(counts.keys()), list(counts.values()),
                    color=['#4CAF50','#2196F3','#FF9800','#F44336','#9C27B0'])
        axes[0].set_title('توزيع الحالات')
        axes[0].tick_params(axis='x', rotation=30)
    else:
        axes[0].text(0.5, 0.5, 'لا بيانات', ha='center')
        axes[0].set_title('توزيع الحالات')

    if CFG.runs_csv.exists():
        runs = pd.read_csv(CFG.runs_csv, encoding='utf-8-sig')
        if not runs.empty:
            vals = pd.to_numeric(runs['words'], errors='coerce').fillna(0)
            axes[1].plot(vals.values, marker='o', color='#2196F3')
            axes[1].set_title('كلمات لكل تشغيل')
        else:
            axes[1].text(0.5, 0.5, 'لا سجلات', ha='center')
            axes[1].set_title('كلمات لكل تشغيل')
    else:
        axes[1].text(0.5, 0.5, 'لا سجلات', ha='center')
        axes[1].set_title('كلمات لكل تشغيل')

    fig_metrics = plot_metrics_fig()
    if fig_metrics:
        plt.close(fig_metrics)
        if CFG.metrics_log.exists():
            mdf = pd.read_csv(CFG.metrics_log, encoding='utf-8-sig')
            if not mdf.empty:
                axes[2].plot(mdf['wer'].dropna().values, label='WER', color='#E53935', marker='o')
                axes[2].plot(mdf['cer'].dropna().values, label='CER', color='#1E88E5', marker='s')
                axes[2].set_title('WER / CER')
                axes[2].legend()
                axes[2].set_ylim(0, 1)
            else:
                axes[2].text(0.5, 0.5, 'لا مقاييس', ha='center')
                axes[2].set_title('WER / CER')
    else:
        axes[2].text(0.5, 0.5, 'لا مقاييس بعد', ha='center')
        axes[2].set_title('WER / CER')

    plt.tight_layout()
    return fig


def get_dashboard_text() -> str:
    lines = ['## 📊 ملخص المشروع']
    counts = DB.count_by_status()
    lines.append(f"**إجمالي الكلمات:** {sum(counts.values())}")
    for k, v in counts.items():
        lines.append(f"- {k}: **{v}**")
    if CFG.stats_json.exists():
        s = json.loads(CFG.stats_json.read_text())
        lines.append(f"\n**آخر تشغيل:** `{s.get('run_id','—')}`  \n"
                     f"صفحات: {s.get('pages')} | كلمات: {s.get('words')} "
                     f"| ثقة: {s.get('avg_confidence')} | مدة: {s.get('duration_sec')}s")
    tail = []
    if _EVENTS.exists():
        with open(_EVENTS, encoding='utf-8', errors='ignore') as f:
            tail = [l.rstrip() for l in f.readlines()[-10:]]
    if tail:
        lines.append('\n**آخر أحداث:**\n```\n' + '\n'.join(tail) + '\n```')
    return '\n'.join(lines)


# --- Gradio action wrappers ---
def do_process(inp, sp, ep, resume, adaptive, progress=gr.Progress()):
    end_page = int(ep) if ep and str(ep).strip() else None
    def cb(cur, tot, msg): progress(cur/tot, desc=msg)
    try:
        stats = process_document(inp, int(sp), end_page, resume, adaptive, cb)
        return f"✅ اكتملت\n```json\n{json.dumps(stats, ensure_ascii=False, indent=2)}\n```"
    except Exception as e:
        return f"❌ {e}"


def do_export():
    s = json.loads(CFG.stats_json.read_text()) if CFG.stats_json.exists() else {}
    rid = s.get('run_id', f'manual_{datetime.now().strftime("%Y%m%d_%H%M%S")}')
    summary = auto_export(rid)
    return f"✅ تصدير اكتمل\n```json\n{json.dumps(summary, ensure_ascii=False, indent=2)}\n```"


def do_backup(): return f"✅ نسخة: `{create_backup()}`"


def do_metrics():
    m = compute_metrics()
    if 'error' in m:
        return m['error'], None
    return (f"**WER:** {m['wer']} | **CER:** {m['cer']} | **عينات:** {m['samples']}",
            build_dashboard_fig())


def do_finetune(min_s, ep, bs, lr, progress=gr.Progress()):
    def cb(cur, tot, msg): progress(cur/tot, desc=msg)
    result = finetune_trocr_lora(int(min_s), int(ep), int(bs), float(lr), cb)
    if 'error' in result:
        return f"❌ {result['error']}"
    return '✅ اكتمل\n' + '\n'.join(result.get('log', []))


def do_al():    return active_learning_cycle()
def do_upload(repo, token): return upload_to_huggingface(repo or None, token or None)

print('✅ Dashboard helpers جاهزة')


✅ Dashboard helpers جاهزة


## الخلية 17: بناء Gradio UI


In [39]:
def build_app():
    with gr.Blocks(title='Arabic Handwriting OCR — Ultimate',
                   theme=gr.themes.Soft(primary_hue='indigo')) as app:

        gr.Markdown(
            '# 🖋️ Arabic Handwriting OCR — النسخة النهائية الموحّدة\n'
            '> TrOCR Batch + Beam Search | Active Learning | WER/CER | Gradio UI')

        # ==================== TAB 1: المعالجة ====================
        with gr.Tab('⚙️ المعالجة'):
            gr.Markdown('### معالجة PDF أو صورة')
            with gr.Row():
                inp_path   = gr.Textbox(label='مسار الملف', value=CFG.pdf_path)
                start_page = gr.Number(label='من الصفحة', value=CFG.pages_start, precision=0)
                end_page   = gr.Textbox(label='إلى الصفحة (فارغ=كل الصفحات)', value=str(CFG.pages_end))
            with gr.Row():
                resume_cb   = gr.Checkbox(label='استئناف من آخر Checkpoint', value=True)
                adaptive_cb = gr.Checkbox(label='Adaptive Threshold', value=False)
            run_btn  = gr.Button('🚀 ابدأ المعالجة', variant='primary')
            run_out  = gr.Markdown()
            run_btn.click(do_process, [inp_path, start_page, end_page, resume_cb, adaptive_cb], run_out)

        # ==================== TAB 2: مراجعة الكلمات ====================
        with gr.Tab('🔍 مراجعة الكلمات'):
            load_w_btn   = gr.Button('📥 تحميل الكلمات')
            word_info    = gr.Markdown('—')
            word_prog    = gr.Textbox(label='التقدم', interactive=False)
            word_img     = gr.Image(label='الصورة', type='pil', height=160)
            word_raw     = gr.Textbox(label='النص الخام', interactive=False)
            word_edit    = gr.Textbox(label='النص المعدّل ✏️')
            conf_slider  = gr.Slider(0, 1, label='الثقة', interactive=False)
            with gr.Row():
                prev_w = gr.Button('⬅ السابق')
                conf_w = gr.Button('✅ تأكيد', variant='primary')
                del_w  = gr.Button('🗑 حذف', variant='stop')
                next_w = gr.Button('التالي ➡')
            _wo = [word_img, word_edit, word_raw, word_info, conf_slider, word_prog]
            load_w_btn.click(load_word_review, outputs=_wo)
            conf_w.click(word_confirm, [word_edit], _wo)
            del_w.click(word_delete, outputs=_wo)
            prev_w.click(word_prev, outputs=_wo)
            next_w.click(word_next, outputs=_wo)

        # ==================== TAB 3: مراجعة الجمل ====================
        with gr.Tab('📝 مراجعة الجمل'):
            load_s_btn  = gr.Button('📥 تحميل الجمل')
            sent_info   = gr.Markdown('—')
            sent_prog   = gr.Textbox(label='التقدم', interactive=False)
            sent_edit   = gr.Textbox(label='الجملة ✏️', lines=3)
            with gr.Row():
                prev_s = gr.Button('⬅ السابق')
                save_s = gr.Button('✅ حفظ', variant='primary')
                next_s = gr.Button('التالي ➡')
            _so = [sent_edit, sent_info, sent_prog]
            load_s_btn.click(load_sent_review, outputs=_so)
            save_s.click(sent_save, [sent_edit], _so)
            prev_s.click(sent_prev, outputs=_so)
            next_s.click(sent_next, outputs=_so)

        # ==================== TAB 4: Dashboard ====================
        with gr.Tab('📊 Dashboard'):
            refresh_btn = gr.Button('🔄 تحديث')
            dash_text   = gr.Markdown()
            dash_plot   = gr.Plot()
            with gr.Row():
                export_btn = gr.Button('📤 تصدير يدوي', variant='secondary')
                backup_btn = gr.Button('💾 نسخة احتياطية', variant='secondary')
                metrics_btn= gr.Button('📐 WER/CER', variant='secondary')
            export_out = gr.Markdown()
            backup_out = gr.Markdown()
            metrics_out= gr.Markdown()
            metrics_plot=gr.Plot()

            refresh_btn.click(lambda: (get_dashboard_text(), build_dashboard_fig()),
                              outputs=[dash_text, dash_plot])
            export_btn.click(do_export, outputs=export_out)
            backup_btn.click(do_backup, outputs=backup_out)
            metrics_btn.click(do_metrics, outputs=[metrics_out, metrics_plot])
            app.load(lambda: (get_dashboard_text(), build_dashboard_fig()),
                     outputs=[dash_text, dash_plot])

        # ==================== TAB 5: Fine-tuning ====================
        with gr.Tab('🧠 Fine-tuning & Active Learning'):
            gr.Markdown('### LoRA Fine-tuning على TrOCR مع TensorBoard')
            with gr.Row():
                ft_min   = gr.Number(label='حد أدنى للعينات', value=CFG.finetune_min_samples, precision=0)
                ft_ep    = gr.Number(label='Epochs', value=CFG.finetune_epochs, precision=0)
                ft_bs    = gr.Number(label='Batch size', value=CFG.finetune_batch_size, precision=0)
                ft_lr    = gr.Number(label='Learning rate', value=CFG.finetune_lr)
            ft_btn  = gr.Button('🔥 ابدأ Fine-tuning', variant='primary')
            ft_out  = gr.Markdown()
            ft_btn.click(do_finetune, [ft_min, ft_ep, ft_bs, ft_lr], ft_out)

            gr.Markdown('---\n### 🔁 Active Learning تلقائي')
            al_info = gr.Markdown(f'الحد الأدنى للتصحيحات: **{CFG.al_min_new_corrections}**')
            al_btn  = gr.Button('🧠 تشغيل دورة Active Learning', variant='secondary')
            al_out  = gr.Markdown()
            al_btn.click(do_al, outputs=al_out)

        # ==================== TAB 6: HuggingFace ====================
        with gr.Tab('🤗 HuggingFace'):
            gr.Markdown('### رفع البيانات إلى HuggingFace Hub')
            hf_repo  = gr.Textbox(label='Repo ID', placeholder='username/my-dataset',
                                   value=CFG.hf_dataset_repo)
            hf_token = gr.Textbox(label='HF Token (اتركه فارغًا لاستخدام Colab Secrets)',
                                   type='password', value='')
            hf_btn   = gr.Button('🚀 رفع إلى Hub', variant='primary')
            hf_out   = gr.Markdown()
            hf_btn.click(do_upload, [hf_repo, hf_token], hf_out)

    return app


## الخلية 18: التشغيل


In [38]:
def launch():
    app = build_app()
    app.launch(
        share=CFG.gradio_share,
        server_port=CFG.gradio_port,
        quiet=False,
    )
    return app


print('✅ كل شيء جاهز!')
print()
print('أوامر التشغيل:')
print('  launch()                     ← فتح Gradio UI كاملة')
print('  process_document()           ← معالجة بدون UI')
print('  auto_export("run_id")        ← تصدير CSV/XLSX/JSONL')
print('  finetune_trocr_lora()        ← LoRA training')
print('  active_learning_cycle()      ← دورة AL تلقائية')
print('  compute_metrics()            ← WER/CER')
print('  create_backup()              ← نسخة احتياطية')


✅ كل شيء جاهز!

أوامر التشغيل:
  launch()                     ← فتح Gradio UI كاملة
  process_document()           ← معالجة بدون UI
  auto_export("run_id")        ← تصدير CSV/XLSX/JSONL
  finetune_trocr_lora()        ← LoRA training
  active_learning_cycle()      ← دورة AL تلقائية
  compute_metrics()            ← WER/CER
  create_backup()              ← نسخة احتياطية


In [40]:
launch()

/tmp/ipykernel_2219/1062931361.py:2: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='Arabic Handwriting OCR — Ultimate',


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e9876015372d4da4ce.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gradio Blocks instance: 18 backend functions
--------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7a810d38fb30>
 |-<gradio.components.number.Number object at 0x7a810d119af0>
 |-<gradio.components.textbox.Textbox object at 0x7a810d1be2d0>
 |-<gradio.components.checkbox.Checkbox object at 0x7a810d327590>
 |-<gradio.components.checkbox.Checkbox object at 0x7a810d4f2570>
 outputs:
 |-<gradio.components.markdown.Markdown object at 0x7a810d4f2c90>
fn_index=1
 inputs:
 outputs:
 |-<gradio.components.image.Image object at 0x7a810ced8530>
 |-<gradio.components.textbox.Textbox object at 0x7a810d14ca70>
 |-<gradio.components.textbox.Textbox object at 0x7a810cf5eb70>
 |-<gradio.components.markdown.Markdown object at 0x7a810d487410>
 |-<gradio.components.slider.Slider object at 0x7a810cdc52b0>
 |-<gradio.components.textbox.Textbox object at 0x7a810ced99d0>
fn_index=2
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7a810d14ca70

In [41]:
# -*- coding: utf-8 -*-
# %% [markdown]
# # 🖋️ Arabic Handwriting OCR — Ultimate (Colab Ready)
# > مع ترحيل تلقائي للبيانات من النسخ السابقة

# %% [bash]
# 📦 تثبيت الاعتماديات (تشغيل مرة واحدة)
!apt-get -y install poppler-utils tesseract-ocr tesseract-ocr-ara -qq
!pip install -q pdf2image easyocr pyspellchecker langdetect transformers peft \
    huggingface_hub datasets Pillow opencv-python-headless pandas numpy \
    matplotlib scikit-learn pytesseract arabic-reshaper python-bidi \
    ar-corrector "gradio>=4.0" openpyxl jiwer torch torchvision \
    tensorboard albumentations tqdm concurrent-log-handler

# %%
# 🔧 الاستيرادات والإعداد
import os, io, gc, cv2, json, time, shutil, sqlite3, logging, platform
import multiprocessing as mp
from pathlib import Path
from dataclasses import dataclass, field
from logging.handlers import RotatingFileHandler
from datetime import datetime
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, torch, gradio as gr, pytesseract, easyocr
from PIL import Image
from pdf2image import convert_from_path
from spellchecker import SpellChecker
from langdetect import detect, DetectorFactory
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from tqdm import tqdm

DetectorFactory.seed = 0

# 🔐 Colab setup
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    _HF_TOKEN = userdata.get('HF_TOKEN') if userdata else None
    if _HF_TOKEN: os.environ['HF_TOKEN'] = _HF_TOKEN
    IN_COLAB = True
    print('✅ Colab: Drive mounted + HF_TOKEN')
except:
    _HF_TOKEN = None
    IN_COLAB = False
    print('ℹ️ Local environment')

# %%
# ⚙️ Config Class (مختصر ومعدل لـ Colab)
@dataclass
class Config:
    project_root: str = '/content/drive/MyDrive/Handwritten_OCR_Ultimate'
    pdf_path: str = '/content/drive/MyDrive/input.pdf'
    db_name: str = 'handwriting_data.db'
    trocr_model_name: str = 'David-Magdy/TR_OCR_LARGE'
    hf_token: str = ''
    hf_dataset_repo: str = ''
    ocr_languages: list = field(default_factory=lambda: ['en', 'ar'])
    dpi: int = 300
    use_gpu: bool = True
    trocr_batch_size: int = 16
    num_beams: int = 4
    max_text_length: int = 64
    easy_conf_threshold: float = 0.80
    trocr_default_conf: float = 0.70
    low_conf_threshold: float = 0.50
    clahe_clip: float = 2.0
    clahe_tile: tuple = (8, 8)
    denoise_h: int = 20
    enable_deskew: bool = True
    min_word_w: int = 15
    min_word_h: int = 10
    dilation_kernel: tuple = (25, 5)
    correction_min_votes: int = 1
    pages_start: int = 1
    pages_end: int = 5
    finetune_min_samples: int = 50
    finetune_epochs: int = 5
    finetune_batch_size: int = 4
    finetune_lr: float = 1e-5
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.1
    lora_targets: list = field(default_factory=lambda: ['query', 'value'])
    al_min_new_corrections: int = 50
    al_reprocess_low_conf_limit: int = 100
    export_val_ratio: float = 0.1
    auto_export_after_run: bool = True
    gradio_share: bool = True
    gradio_port: int = 7860
    log_level: str = 'INFO'

    @property
    def root(self): return Path(self.project_root)
    @property
    def db_path(self): return self.root / 'database' / self.db_name
    @property
    def logs_dir(self): return self.root / 'logs'
    @property
    def exports_dir(self): return self.root / 'exports'
    @property
    def cache_dir(self): return self.root / 'models_cache'
    @property
    def artifacts_dir(self): return self.root / 'artifacts'
    @property
    def backups_dir(self): return self.root / 'backups'
    @property
    def tensorboard_dir(self): return self.root / 'runs'
    @property
    def feedback_csv(self): return self.logs_dir / 'user_corrections_feedback.csv'
    @property
    def stats_json(self): return self.logs_dir / 'processing_stats.json'
    @property
    def correction_dict_path(self): return self.artifacts_dir / 'correction_dict.json'
    @property
    def checkpoint_file(self): return self.artifacts_dir / 'ocr_checkpoint.json'
    @property
    def metrics_log(self): return self.logs_dir / 'metrics_history.csv'
    @property
    def runs_csv(self): return self.logs_dir / 'runs_history.csv'
    @property
    def lora_save_path(self): return self.cache_dir / 'trocr_lora_finetuned'
    @property
    def easyocr_drive_path(self): return self.root / '.EasyOCR'

    def setup(self):
        for d in [self.root/'database', self.logs_dir, self.exports_dir,
                  self.cache_dir, self.artifacts_dir, self.backups_dir, self.tensorboard_dir]:
            d.mkdir(parents=True, exist_ok=True)
        if self.hf_token: os.environ['HF_TOKEN'] = self.hf_token
        if self.cache_dir.exists():
            os.environ['TRANSFORMERS_CACHE'] = str(self.cache_dir)
            os.environ['TORCH_HOME'] = str(self.cache_dir)
        for csv_path, cols in [
            (self.feedback_csv, ['timestamp','image_id','original_text','corrected_text','status']),
            (self.runs_csv, ['run_id','timestamp','pages','words','avg_conf','duration_sec','status']),
        ]:
            if not csv_path.exists():
                pd.DataFrame(columns=cols).to_csv(csv_path, index=False, encoding='utf-8-sig')

    @classmethod
    def for_colab(cls, pdf_name='input.pdf', hf_token='', hf_repo=''):
        return cls(
            project_root='/content/drive/MyDrive/Handwritten_OCR_Ultimate',
            pdf_path=f'/content/drive/MyDrive/{pdf_name}',
            hf_token=hf_token or _HF_TOKEN or '',
            hf_dataset_repo=hf_repo,
        )

CFG = Config.for_colab()
CFG.setup()
print(f'✅ Config جاهز: {CFG.root}')

# %%
# 🔄 دالة ترحيل البيانات من النسخ القديمة (مهمة!)
def migrate_old_data(base_path='/content/drive/MyDrive'):
    """ترحيل قواعد البيانات والتصححات من المجلدات القديمة"""
    old_folders = ['Handwriting_Dataset', 'Handwritten_OCR_Integrated', 'Handwritten_OCR_Pro']
    target_db = CFG.db_path
    target_feedback = CFG.feedback_csv

    print('🔍 البحث عن بيانات قديمة...')
    migrated = {'dbs': 0, 'feedback': 0}

    # دمج قواعد البيانات
    for folder in old_folders:
        folder_path = Path(base_path) / folder
        if not folder_path.exists(): continue
        for db_file in folder_path.rglob('*.db'):
            try:
                src_conn = sqlite3.connect(db_file)
                tgt_conn = sqlite3.connect(target_db)
                src_cur = src_conn.cursor()
                tgt_cur = tgt_conn.cursor()

                # نسخ البيانات الموثقة فقط
                src_cur.execute("""
                    SELECT image_data, predicted_text, raw_text, status, confidence,
                           model_source, x, y, w, h, page_num, created_at
                    FROM handwriting_data WHERE status IN ('verified','sentence_corrected','yes')
                """)
                rows = src_cur.fetchall()

                for row in rows:
                    try:
                        tgt_cur.execute("""
                            INSERT INTO handwriting_data
                            (image_data, predicted_text, raw_text, status, confidence,
                             model_source, x, y, w, h, page_num, run_id, created_at, updated_at)
                            VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
                        """, (*row, f'migrated_{folder}', row[-1] if len(row)>11 else '', row[-1] if len(row)>11 else ''))
                        migrated['dbs'] += 1
                    except: continue
                tgt_conn.commit()
                src_conn.close(); tgt_conn.close()
                print(f'  📥 من {folder}: {migrated["dbs"]} سجل')
            except Exception as e:
                print(f'  ⚠️ خطأ في {db_file}: {e}')

    # دمج ملفات التصحيحات
    all_feedback = []
    for folder in old_folders:
        folder_path = Path(base_path) / folder
        if not folder_path.exists(): continue
        for fb_file in folder_path.rglob('*feedback*.csv'):
            try:
                df = pd.read_csv(fb_file, encoding='utf-8-sig')
                all_feedback.append(df)
                migrated['feedback'] += len(df)
            except: continue

    if all_feedback:
        merged = pd.concat(all_feedback, ignore_index=True).drop_duplicates(
            subset=['original_text','corrected_text'], keep='last')
        merged.to_csv(target_feedback, index=False, encoding='utf-8-sig')
        print(f'✅ التصحيحات المدمجة: {len(merged)} سجل')

    # إعادة بناء قاموس التصحيح
    if target_feedback.exists():
        df = pd.read_csv(target_feedback, encoding='utf-8-sig')
        buckets = defaultdict(Counter)
        for _, row in df.iterrows():
            o, c = str(row.get('original_text','')).strip(), str(row.get('corrected_text','')).strip()
            if o and c and o != c: buckets[o][c] += 1
        correction_dict = {o: cnt.most_common(1)[0][0] for o, cnt in buckets.items() if cnt.most_common(1)[0][1] >= 1}
        CFG.correction_dict_path.write_text(json.dumps(correction_dict, ensure_ascii=False, indent=2), 'utf-8')
        print(f'✅ قاموس التصحيح: {len(correction_dict)} إدخال')

    print(f'🎯 الترحيل اكتمل: {migrated["dbs"]} كلمة + {migrated["feedback"]} تصحيح')
    return migrated

# %%
# 🚀 تشغيل الترحيل (اختياري - شغله مرة واحدة)
# migrate_old_data()  # ← احذف # لتفعيل الترحيل

# %%
# 📊 بقية الكود (قاعدة البيانات، المعالجة، Gradio...)
# [نفس الكود الأصلي من خلية 4 فما بعد - انسخه كما هو]
# ... (باقي الكود الطويل) ...

# %%
# ▶️ التشغيل النهائي
def launch():
    app = build_app()  # تأكد من وجود دالة build_app() في الكود
    app.launch(share=CFG.gradio_share, server_port=CFG.gradio_port, quiet=False)
    return app

# لتشغيل الواجهة:
# launch()

# أو للمعالجة المباشرة بدون UI:
# stats = process_document()
# print(stats)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Colab: Drive mounted + HF_TOKEN
✅ Config جاهز: /content/drive/MyDrive/Handwritten_OCR_Ultimate


In [42]:
# احذف علامة # من السطر التالي لترحيل البيانات القديمة:
migrate_old_data()

🔍 البحث عن بيانات قديمة...
  ⚠️ خطأ في /content/drive/MyDrive/Handwriting_Dataset/handwriting_data.db: no such column: raw_text
  📥 من Handwritten_OCR_Pro: 0 سجل
  📥 من Handwritten_OCR_Pro: 0 سجل
✅ التصحيحات المدمجة: 354 سجل
✅ قاموس التصحيح: 262 إدخال
🎯 الترحيل اكتمل: 0 كلمة + 475 تصحيح


/tmp/ipykernel_2219/4054377592.py:211: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged = pd.concat(all_feedback, ignore_index=True).drop_duplicates(


{'dbs': 0, 'feedback': 475}